## 1. Doc du lieu (7 file CSV)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

spark = SparkSession.builder \
    .appName("HomeCreditCleaning") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

DATA_DIR = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data"

TABLE_FILES = {
    "app_train":    "application_train.csv",
    "app_test":     "application_test.csv",
    "bureau":       "bureau.csv",
    "bureau_bal":   "bureau_balance.csv",
    "prev_app":     "previous_application.csv",
    "pos_cash":     "POS_CASH_balance.csv",
    "credit_card":  "credit_card_balance.csv",
    "installments": "installments_payments.csv",
}

dfs = {}

print(f"{'Table':<15} {'Rows':>12} {'Cols':>6} {'File':<35}")
print("=" * 70)

for name, fname in TABLE_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    df = spark.read.csv(path, header=True, inferSchema=True, nullValue="XNA")
    dfs[name] = df
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{name:<15} {row_count:>12,} {col_count:>6} {fname:<35}")

print("=" * 70)
print(f"Total: {len(dfs)} tables loaded")

Table                   Rows   Cols File                               
app_train            307,511    122 application_train.csv              
app_test              48,744    121 application_test.csv               
bureau             1,716,428     17 bureau.csv                         
bureau_bal        27,299,925      3 bureau_balance.csv                 
prev_app           1,670,214     37 previous_application.csv           
pos_cash          10,001,358      8 POS_CASH_balance.csv               
credit_card        3,840,312     23 credit_card_balance.csv            
installments      13,605,401      8 installments_payments.csv          
Total: 8 tables loaded


## 2. Phan tich Missing Values tung bang

Voi moi bang:
- Tinh ty le missing cua tung cot
- De xuat chien luoc: **Drop** (>60%), **Median** (numeric), **Mode** (categorical)
- Trinh bay duoi dang bang ro rang

In [6]:
def analyze_missing(df, name):
    """Phan tich missing values va de xuat xu ly cho tung cot (single-pass)."""
    total = df.count()
    dtypes_dict = dict(df.dtypes)

    # Single-pass: count nulls for ALL columns at once (thay vi N lan scan)
    null_exprs = [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
    null_row = df.select(null_exprs).collect()[0]

    results = []
    for col_name in df.columns:
        nc = int(null_row[col_name])
        if nc == 0:
            continue
        dtype = dtypes_dict[col_name]
        pct = round(nc / total * 100, 2)

        if pct > 60:
            strategy = "DROP column (>60%)"
        elif dtype in ("int", "double", "float", "bigint", "long"):
            strategy = "Fill MEDIAN"
        else:
            strategy = "Fill MODE"

        results.append((col_name, dtype, nc, pct, strategy))

    if results:
        results.sort(key=lambda x: -x[3])
        print(f"\n{'='*95}")
        print(f"  MISSING VALUES: {name} ({total:,} rows, {len(df.columns)} cols)")
        print(f"  Columns with missing: {len(results)} / {len(df.columns)}")
        print(f"{'='*95}")
        print(f"  {'Column':<42} {'Type':<10} {'Null_Count':>12} {'Null_%':>8}  {'Strategy':<20}")
        print(f"  {'-'*92}")
        for col_name, dtype, nc, pct, strategy in results:
            print(f"  {col_name:<42} {dtype:<10} {nc:>12,} {pct:>7.2f}%  {strategy}")
    else:
        print(f"\n[{name}] Khong co missing values!")

    return results

In [7]:
missing_reports = {}
for name, df in dfs.items():
    missing_reports[name] = analyze_missing(df, name)


  MISSING VALUES: app_train (307,511 rows, 122 cols)
  Columns with missing: 69 / 122
  Column                                     Type         Null_Count   Null_%  Strategy            
  --------------------------------------------------------------------------------------------
  COMMONAREA_AVG                             double          214,865   69.87%  DROP column (>60%)
  COMMONAREA_MODE                            double          214,865   69.87%  DROP column (>60%)
  COMMONAREA_MEDI                            double          214,865   69.87%  DROP column (>60%)
  NONLIVINGAPARTMENTS_AVG                    double          213,514   69.43%  DROP column (>60%)
  NONLIVINGAPARTMENTS_MODE                   double          213,514   69.43%  DROP column (>60%)
  NONLIVINGAPARTMENTS_MEDI                   double          213,514   69.43%  DROP column (>60%)
  FONDKAPREMONT_MODE                         string          210,295   68.39%  DROP column (>60%)
  LIVINGAPARTMENTS_AVG          

## 3. Phat hien & Xu ly Anomalies tung bang

Cac anomaly can phat hien:
- `DAYS_EMPLOYED = 365243` -> gia tri sentinel, chuyen thanh Null
- Cac cot `DAYS_*` co gia tri duong bat thuong (binh thuong phai am)
- Cac gia tri outlier cuc doan trong cot `AMT_*`

In [8]:
print("=" * 90)
print("  QUET ANOMALY TREN TAT CA CAC BANG")
print("=" * 90)

# --- 1. DAYS_EMPLOYED = 365243 (sentinel value) ---
print("\n--- DAYS_EMPLOYED = 365243 (sentinel value) ---")
for tbl in ["app_train", "app_test"]:
    if tbl in dfs and "DAYS_EMPLOYED" in dfs[tbl].columns:
        row = dfs[tbl].select(
            F.sum((F.col("DAYS_EMPLOYED") == 365243).cast("int")).alias("cnt"),
            F.count("*").alias("total")
        ).collect()[0]
        print(f"  [{tbl}] DAYS_EMPLOYED == 365243: {row['cnt']:,} / {row['total']:,} ({row['cnt']/row['total']*100:.1f}%)")

# --- 2. DAYS_* cot co gia tri duong (phai la am) - batch per table ---
print("\n--- DAYS_* columns voi gia tri duong bat thuong ---")
for tbl_name in ["app_train", "app_test", "bureau", "prev_app", "installments"]:
    if tbl_name not in dfs:
        continue
    df = dfs[tbl_name]
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if not days_cols:
        continue
    exprs = [F.sum((F.col(c) > 0).cast("int")).alias(c) for c in days_cols]
    row = df.select(exprs).collect()[0]
    for col in days_cols:
        cnt = int(row[col])
        if cnt > 0:
            print(f"  [{tbl_name}] {col}: {cnt:,} gia tri duong")

# --- 3. DAYS_* = 365243 trong previous_application - batch ---
print("\n--- DAYS_* = 365243 trong previous_application ---")
if "prev_app" in dfs:
    df = dfs["prev_app"]
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    exprs = [F.sum((F.col(c) == 365243).cast("int")).alias(c) for c in days_cols]
    exprs.append(F.count("*").alias("_total"))
    row = df.select(exprs).collect()[0]
    total = row["_total"]
    for col in days_cols:
        cnt = int(row[col])
        if cnt > 0:
            print(f"  [prev_app] {col} == 365243: {cnt:,} ({cnt/total*100:.1f}%)")

# --- 4. AMT_* pham vi gia tri - batch per table ---
print("\n--- AMT_* pham vi gia tri (kiem tra outlier cuc doan) ---")
for tbl_name in ["app_train", "prev_app", "installments", "credit_card"]:
    if tbl_name not in dfs:
        continue
    df = dfs[tbl_name]
    amt_cols = [c for c in df.columns if c.startswith("AMT_")]
    if not amt_cols:
        continue
    exprs = []
    for col in amt_cols:
        exprs.extend([
            F.min(col).alias(f"{col}_min"),
            F.max(col).alias(f"{col}_max"),
            F.mean(col).alias(f"{col}_mean"),
        ])
    row = df.select(exprs).collect()[0]
    print(f"\n  [{tbl_name}]")
    for col in amt_cols:
        mn, mx, avg = row[f"{col}_min"], row[f"{col}_max"], row[f"{col}_mean"]
        if mn is not None:
            print(f"    {col:35s} min={mn:>15,.2f}  max={mx:>15,.2f}  mean={avg:>15,.2f}")

  QUET ANOMALY TREN TAT CA CAC BANG

--- DAYS_EMPLOYED = 365243 (sentinel value) ---
  [app_train] DAYS_EMPLOYED == 365243: 55,374 / 307,511 (18.0%)
  [app_test] DAYS_EMPLOYED == 365243: 9,274 / 48,744 (19.0%)

--- DAYS_* columns voi gia tri duong bat thuong ---
  [app_train] DAYS_EMPLOYED: 55,374 gia tri duong
  [app_test] DAYS_EMPLOYED: 9,274 gia tri duong
  [bureau] DAYS_CREDIT_ENDDATE: 602,603 gia tri duong
  [bureau] DAYS_CREDIT_UPDATE: 17 gia tri duong
  [prev_app] DAYS_FIRST_DRAWING: 934,444 gia tri duong
  [prev_app] DAYS_FIRST_DUE: 40,645 gia tri duong
  [prev_app] DAYS_LAST_DUE_1ST_VERSION: 318,256 gia tri duong
  [prev_app] DAYS_LAST_DUE: 211,221 gia tri duong
  [prev_app] DAYS_TERMINATION: 225,913 gia tri duong

--- DAYS_* = 365243 trong previous_application ---
  [prev_app] DAYS_FIRST_DRAWING == 365243: 934,444 (55.9%)
  [prev_app] DAYS_FIRST_DUE == 365243: 40,645 (2.4%)
  [prev_app] DAYS_LAST_DUE_1ST_VERSION == 365243: 93,864 (5.6%)
  [prev_app] DAYS_LAST_DUE == 365243: 2

## 4. Xu ly tung bang doc lap (truoc khi join)

### 4.1 application_train / application_test
- Xu ly anomalies: `DAYS_EMPLOYED=365243` -> Null, `DAYS_*` duong -> am
- Xu ly missing values: median (numeric), mode (categorical)
- Loai bo cac cot co > 60% gia tri thieu
- IQR capping cho cot tai chinh quan trong
- `CNT_CHILDREN` cap [0, 10], `OWN_CAR_AGE` cap [0, 100]

In [9]:
def clean_application(df, name):
    """Lam sach application_train hoac application_test."""
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: {name} ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    # --- Buoc 1: DAYS_EMPLOYED = 365243 -> Null ---
    if "DAYS_EMPLOYED" in df.columns:
        cnt = df.select(F.sum((F.col("DAYS_EMPLOYED") == 365243).cast("int"))).collect()[0][0]
        df = df.withColumn(
            "DAYS_EMPLOYED",
            F.when(F.col("DAYS_EMPLOYED") == 365243, None).otherwise(F.col("DAYS_EMPLOYED"))
        )
        print(f"  [Anomaly] DAYS_EMPLOYED=365243 -> Null: {cnt:,} rows")

    # --- Buoc 2: DAYS_* duong -> am hoa (batch check) ---
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if days_cols:
        pos_exprs = [F.sum((F.col(c) > 0).cast("int")).alias(c) for c in days_cols]
        pos_row = df.select(pos_exprs).collect()[0]
        for col in days_cols:
            pos_count = int(pos_row[col])
            if pos_count > 0:
                df = df.withColumn(col, F.when(F.col(col) > 0, -F.col(col)).otherwise(F.col(col)))
                print(f"  [Anomaly] {col}: {pos_count:,} positive -> negated")

    # --- Buoc 3: Drop cot co >60% missing (single-pass null count) ---
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_row = df.select(null_exprs).collect()[0]

    cols_to_drop = [c for c in df.columns if int(null_row[c]) / total > 0.60]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        print(f"  [Missing] Dropped {len(cols_to_drop)} cols (>60% null):")
        for c in cols_to_drop:
            print(f"            - {c}")

    # --- Buoc 4: Fill missing - median (numeric), mode (categorical) ---
    # Re-scan nulls after dropping columns (single pass)
    null_exprs2 = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_row2 = df.select(null_exprs2).collect()[0]

    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    string_cols  = [c for c, t in df.dtypes if t == "string"]

    num_with_null = [c for c in numeric_cols if int(null_row2[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {col: medians[i][0] for i, col in enumerate(num_with_null) if medians[i]}
        df = df.fillna(fill_map)
    print(f"  [Missing] Filled {len(num_with_null)} numeric cols (median)")

    filled_cat = 0
    cat_with_null = [c for c in string_cols if int(null_row2[c]) > 0]
    for col in cat_with_null:
        mode_row = df.groupBy(col).count().orderBy(F.desc("count")).first()
        if mode_row and mode_row[col]:
            df = df.fillna({col: mode_row[col]})
            filled_cat += 1
    print(f"  [Missing] Filled {filled_cat} categorical cols (mode)")

    # --- Buoc 5: IQR capping (3x) cho cot tai chinh ---
    cap_cols = [c for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"] if c in df.columns]
    if cap_cols:
        quantiles = df.approxQuantile(cap_cols, [0.25, 0.75], 0.001)
        for i, col in enumerate(cap_cols):
            q1, q3 = quantiles[i][0], quantiles[i][1]
            iqr = q3 - q1
            lower, upper = q1 - 3.0 * iqr, q3 + 3.0 * iqr
            df = df.withColumn(
                col,
                F.when(F.col(col) < lower, lower)
                 .when(F.col(col) > upper, upper)
                 .otherwise(F.col(col))
            )
            print(f"  [Outlier] {col}: capped to [{lower:,.0f}, {upper:,.0f}]")

    # --- Buoc 6: CNT_CHILDREN cap [0, 10] ---
    if "CNT_CHILDREN" in df.columns:
        df = df.withColumn(
            "CNT_CHILDREN",
            F.when((F.col("CNT_CHILDREN") < 0) | (F.col("CNT_CHILDREN") > 10), 0)
             .otherwise(F.col("CNT_CHILDREN"))
        )
        print(f"  [Anomaly] CNT_CHILDREN: capped to [0, 10]")

    # --- Buoc 7: OWN_CAR_AGE cap [0, 100] ---
    if "OWN_CAR_AGE" in df.columns:
        df = df.withColumn(
            "OWN_CAR_AGE",
            F.when((F.col("OWN_CAR_AGE") < 0) | (F.col("OWN_CAR_AGE") > 100), None)
             .otherwise(F.col("OWN_CAR_AGE"))
        )
        median_car = df.approxQuantile("OWN_CAR_AGE", [0.5], 0.001)[0]
        df = df.fillna({"OWN_CAR_AGE": median_car})
        print(f"  [Anomaly] OWN_CAR_AGE: capped to [0, 100], fill null -> median={median_car}")

    # --- Kiem tra ket qua (single-pass) ---
    final_nulls = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["app_train"] = clean_application(dfs["app_train"], "application_train")
dfs["app_test"]  = clean_application(dfs["app_test"], "application_test")


  CLEANING: application_train (307,511 rows x 122 cols)
  [Anomaly] DAYS_EMPLOYED=365243 -> Null: 55,374 rows
  [Missing] Dropped 17 cols (>60% null):
            - OWN_CAR_AGE
            - YEARS_BUILD_AVG
            - COMMONAREA_AVG
            - FLOORSMIN_AVG
            - LIVINGAPARTMENTS_AVG
            - NONLIVINGAPARTMENTS_AVG
            - YEARS_BUILD_MODE
            - COMMONAREA_MODE
            - FLOORSMIN_MODE
            - LIVINGAPARTMENTS_MODE
            - NONLIVINGAPARTMENTS_MODE
            - YEARS_BUILD_MEDI
            - COMMONAREA_MEDI
            - FLOORSMIN_MEDI
            - LIVINGAPARTMENTS_MEDI
            - NONLIVINGAPARTMENTS_MEDI
            - FONDKAPREMONT_MODE
  [Missing] Filled 46 numeric cols (median)
  [Missing] Filled 4 categorical cols (mode)
  [Outlier] AMT_INCOME_TOTAL: capped to [-157,500, 472,500]
  [Outlier] AMT_CREDIT: capped to [-1,345,950, 2,424,600]
  [Outlier] AMT_ANNUITY: capped to [-37,737, 88,830]
  [Outlier] AMT_GOODS_PRICE: capped to 

### 4.2 bureau
- Xu ly missing values (median/mode)
- Xu ly anomalies: `DAYS_*` duong -> am, `AMT_*` am (ghi nhan)

In [10]:
def clean_bureau(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: bureau ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    # --- Anomaly: DAYS_* duong -> am (batch) ---
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if days_cols:
        pos_row = df.select([F.sum((F.col(c) > 0).cast("int")).alias(c) for c in days_cols]).collect()[0]
        for col in days_cols:
            pos_count = int(pos_row[col])
            if pos_count > 0:
                df = df.withColumn(col, F.when(F.col(col) > 0, -F.col(col)).otherwise(F.col(col)))
                print(f"  [Anomaly] {col}: {pos_count:,} positive -> negated")

    # --- Ghi nhan AMT_* am (batch) ---
    amt_cols = [c for c in df.columns if c.startswith("AMT_")]
    if amt_cols:
        neg_row = df.select([F.sum((F.col(c) < 0).cast("int")).alias(c) for c in amt_cols]).collect()[0]
        for col in amt_cols:
            neg_count = int(neg_row[col])
            if neg_count > 0:
                print(f"  [Note] {col}: {neg_count:,} negative values (giu nguyen - co the la du no)")

    # --- Drop cot >60% null (single-pass) ---
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    cols_to_drop = [c for c in df.columns if int(null_row[c]) / total > 0.60]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        print(f"  [Missing] Dropped {len(cols_to_drop)} cols (>60% null): {cols_to_drop}")

    # --- Fill missing (single-pass null check + batch median) ---
    null_row2 = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    string_cols  = [c for c, t in df.dtypes if t == "string"]

    num_with_null = [c for c in numeric_cols if int(null_row2[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {col: medians[i][0] for i, col in enumerate(num_with_null) if medians[i]}
        df = df.fillna(fill_map)

    filled_cat = 0
    for col in string_cols:
        if int(null_row2[col]) > 0:
            mode_row = df.groupBy(col).count().orderBy(F.desc("count")).first()
            if mode_row and mode_row[col]:
                df = df.fillna({col: mode_row[col]})
                filled_cat += 1

    print(f"  [Missing] Filled {len(num_with_null)} numeric (median), {filled_cat} categorical (mode)")

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["bureau"] = clean_bureau(dfs["bureau"])


  CLEANING: bureau (1,716,428 rows x 17 cols)
  [Anomaly] DAYS_CREDIT_ENDDATE: 602,603 positive -> negated
  [Anomaly] DAYS_CREDIT_UPDATE: 17 positive -> negated
  [Note] AMT_CREDIT_SUM_DEBT: 8,418 negative values (giu nguyen - co the la du no)
  [Note] AMT_CREDIT_SUM_LIMIT: 351 negative values (giu nguyen - co the la du no)
  [Missing] Dropped 2 cols (>60% null): ['AMT_CREDIT_MAX_OVERDUE', 'AMT_ANNUITY']
  [Missing] Filled 5 numeric (median), 0 categorical (mode)

  => Ket qua: 1,716,428 rows x 15 cols | Null con lai: 0


### 4.3 bureau_balance
- Xu ly missing values (cot STATUS)
- Hien thi phan phoi STATUS truoc va sau cleaning

In [11]:
def clean_bureau_balance(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: bureau_balance ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    # --- Phan phoi STATUS truoc cleaning ---
    print("\n  Phan phoi STATUS truoc cleaning:")
    df.groupBy("STATUS").count().orderBy(F.desc("count")).show()

    # --- Single-pass null check ---
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]

    # --- Fill missing STATUS bang mode ---
    status_nulls = int(null_row["STATUS"])
    if status_nulls > 0:
        mode_row = df.groupBy("STATUS").count().orderBy(F.desc("count")).first()
        if mode_row:
            df = df.fillna({"STATUS": mode_row["STATUS"]})
            print(f"  [Missing] STATUS: filled {status_nulls:,} nulls voi mode = '{mode_row['STATUS']}'")
    else:
        print(f"  [Missing] STATUS: khong co null")

    # --- Fill missing cho cac cot numeric khac (batch) ---
    other_num = [c for c in df.columns if c != "STATUS" and dict(df.dtypes)[c] in ("int", "double", "float", "bigint", "long")]
    num_with_null = [c for c in other_num if int(null_row[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {col: medians[i][0] for i, col in enumerate(num_with_null) if medians[i]}
        df = df.fillna(fill_map)
        for col in num_with_null:
            print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls voi median")

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["bureau_bal"] = clean_bureau_balance(dfs["bureau_bal"])


  CLEANING: bureau_balance (27,299,925 rows x 3 cols)

  Phan phoi STATUS truoc cleaning:
+------+--------+
|STATUS|   count|
+------+--------+
|     C|13646993|
|     0| 7499507|
|     X| 5810482|
|     1|  242347|
|     5|   62406|
|     2|   23419|
|     3|    8924|
|     4|    5847|
+------+--------+

  [Missing] STATUS: khong co null

  => Ket qua: 27,299,925 rows x 3 cols | Null con lai: 0


### 4.4 previous_application
- Xu ly anomalies: `DAYS_*` = 365243 -> Null, `DAYS_*` duong -> am
- Xu ly missing values (median/mode)
- Loai bo cot >60% missing

In [12]:
def clean_previous_application(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: previous_application ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    # --- Anomaly: DAYS_* = 365243 -> Null (batch check) ---
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if days_cols:
        sentinel_row = df.select([F.sum((F.col(c) == 365243).cast("int")).alias(c) for c in days_cols]).collect()[0]
        for col in days_cols:
            cnt = int(sentinel_row[col])
            if cnt > 0:
                df = df.withColumn(col, F.when(F.col(col) == 365243, None).otherwise(F.col(col)))
                print(f"  [Anomaly] {col}=365243 -> Null: {cnt:,} rows")

    # --- Anomaly: DAYS_* duong -> am (batch check) ---
    if days_cols:
        pos_row = df.select([F.sum((F.col(c) > 0).cast("int")).alias(c) for c in days_cols]).collect()[0]
        for col in days_cols:
            pos_count = int(pos_row[col])
            if pos_count > 0:
                df = df.withColumn(col, F.when(F.col(col) > 0, -F.col(col)).otherwise(F.col(col)))
                print(f"  [Anomaly] {col}: {pos_count:,} positive -> negated")

    # --- Drop cot >60% missing (single-pass) ---
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    cols_to_drop = [c for c in df.columns if int(null_row[c]) / total > 0.60]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        print(f"  [Missing] Dropped {len(cols_to_drop)} cols (>60% null):")
        for c in cols_to_drop:
            print(f"            - {c}")

    # --- Fill missing (batch) ---
    null_row2 = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    string_cols  = [c for c, t in df.dtypes if t == "string"]

    num_with_null = [c for c in numeric_cols if int(null_row2[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {col: medians[i][0] for i, col in enumerate(num_with_null) if medians[i]}
        df = df.fillna(fill_map)

    filled_cat = 0
    for col in string_cols:
        if int(null_row2[col]) > 0:
            mode_row = df.groupBy(col).count().orderBy(F.desc("count")).first()
            if mode_row and mode_row[col]:
                df = df.fillna({col: mode_row[col]})
                filled_cat += 1

    print(f"  [Missing] Filled {len(num_with_null)} numeric (median), {filled_cat} categorical (mode)")

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["prev_app"] = clean_previous_application(dfs["prev_app"])


  CLEANING: previous_application (1,670,214 rows x 37 cols)
  [Anomaly] DAYS_FIRST_DRAWING=365243 -> Null: 934,444 rows
  [Anomaly] DAYS_FIRST_DUE=365243 -> Null: 40,645 rows
  [Anomaly] DAYS_LAST_DUE_1ST_VERSION=365243 -> Null: 93,864 rows
  [Anomaly] DAYS_LAST_DUE=365243 -> Null: 211,221 rows
  [Anomaly] DAYS_TERMINATION=365243 -> Null: 225,913 rows
  [Anomaly] DAYS_LAST_DUE_1ST_VERSION: 224,392 positive -> negated
  [Missing] Dropped 4 cols (>60% null):
            - RATE_INTEREST_PRIMARY
            - RATE_INTEREST_PRIVILEGED
            - NAME_PRODUCT_TYPE
            - DAYS_FIRST_DRAWING
  [Missing] Filled 11 numeric (median), 7 categorical (mode)

  => Ket qua: 1,670,214 rows x 33 cols | Null con lai: 3,144,149


### 4.5 POS_CASH_balance
- Xu ly missing values (median/mode)

In [13]:
def clean_pos_cash(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: POS_CASH_balance ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]

    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    string_cols  = [c for c, t in df.dtypes if t == "string"]

    num_with_null = [c for c in numeric_cols if int(null_row[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {}
        for i, col in enumerate(num_with_null):
            if medians[i]:
                fill_map[col] = medians[i][0]
                print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls (median = {medians[i][0]})")
        df = df.fillna(fill_map)

    filled_cat = 0
    for col in string_cols:
        if int(null_row[col]) > 0:
            mode_row = df.groupBy(col).count().orderBy(F.desc("count")).first()
            if mode_row and mode_row[col]:
                df = df.fillna({col: mode_row[col]})
                print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls (mode = '{mode_row[col]}')")
                filled_cat += 1

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["pos_cash"] = clean_pos_cash(dfs["pos_cash"])


  CLEANING: POS_CASH_balance (10,001,358 rows x 8 cols)
  [Missing] CNT_INSTALMENT: filled 26,071 nulls (median = 12.0)
  [Missing] CNT_INSTALMENT_FUTURE: filled 26,087 nulls (median = 7.0)
  [Missing] NAME_CONTRACT_STATUS: filled 2 nulls (mode = 'Active')

  => Ket qua: 10,001,358 rows x 8 cols | Null con lai: 0


### 4.6 installments_payments
- Xu ly anomalies: `AMT_PAYMENT` am -> Null, `DAYS_*` duong -> am
- Xu ly missing values (median)

In [14]:
def clean_installments(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: installments_payments ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    # --- Batch anomaly check ---
    check_exprs = []
    amt_neg_cols = [c for c in ["AMT_PAYMENT", "AMT_INSTALMENT"] if c in df.columns]
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    for c in amt_neg_cols:
        check_exprs.append(F.sum((F.col(c) < 0).cast("int")).alias(f"{c}_neg"))
    for c in days_cols:
        check_exprs.append(F.sum((F.col(c) > 0).cast("int")).alias(f"{c}_pos"))
    if check_exprs:
        anom_row = df.select(check_exprs).collect()[0]

    # --- Anomaly: AMT_* am -> Null ---
    for col in amt_neg_cols:
        neg_count = int(anom_row[f"{col}_neg"])
        if neg_count > 0:
            df = df.withColumn(col, F.when(F.col(col) < 0, None).otherwise(F.col(col)))
            print(f"  [Anomaly] {col}: {neg_count:,} negative -> Null")

    # --- Anomaly: DAYS_* duong -> am ---
    for col in days_cols:
        pos_count = int(anom_row[f"{col}_pos"])
        if pos_count > 0:
            df = df.withColumn(col, F.when(F.col(col) > 0, -F.col(col)).otherwise(F.col(col)))
            print(f"  [Anomaly] {col}: {pos_count:,} positive -> negated")

    # --- Fill missing (batch) ---
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    num_with_null = [c for c in numeric_cols if int(null_row[c]) > 0]

    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {}
        for i, col in enumerate(num_with_null):
            if medians[i]:
                fill_map[col] = medians[i][0]
                print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls (median = {medians[i][0]})")
        df = df.fillna(fill_map)

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["installments"] = clean_installments(dfs["installments"])


  CLEANING: installments_payments (13,605,401 rows x 8 cols)
  [Missing] DAYS_ENTRY_PAYMENT: filled 2,905 nulls (median = -827.0)
  [Missing] AMT_PAYMENT: filled 2,905 nulls (median = 8116.74)

  => Ket qua: 13,605,401 rows x 8 cols | Null con lai: 0


### 4.7 credit_card_balance
- Xu ly missing values (median/mode)

In [15]:
def clean_credit_card(df):
    total = df.count()
    print(f"\n{'='*70}")
    print(f"  CLEANING: credit_card_balance ({total:,} rows x {len(df.columns)} cols)")
    print(f"{'='*70}")

    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]

    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint", "long")]
    string_cols  = [c for c, t in df.dtypes if t == "string"]

    num_with_null = [c for c in numeric_cols if int(null_row[c]) > 0]
    if num_with_null:
        medians = df.approxQuantile(num_with_null, [0.5], 0.001)
        fill_map = {}
        for i, col in enumerate(num_with_null):
            if medians[i]:
                fill_map[col] = medians[i][0]
                print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls (median = {medians[i][0]})")
        df = df.fillna(fill_map)

    filled_cat = 0
    for col in string_cols:
        if int(null_row[col]) > 0:
            mode_row = df.groupBy(col).count().orderBy(F.desc("count")).first()
            if mode_row and mode_row[col]:
                df = df.fillna({col: mode_row[col]})
                print(f"  [Missing] {col}: filled {int(null_row[col]):,} nulls (mode = '{mode_row[col]}')")
                filled_cat += 1

    final_nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    remaining = sum(int(final_nulls[c]) for c in df.columns)
    print(f"\n  => Ket qua: {df.count():,} rows x {len(df.columns)} cols | Null con lai: {remaining:,}")
    return df

dfs["credit_card"] = clean_credit_card(dfs["credit_card"])


  CLEANING: credit_card_balance (3,840,312 rows x 23 cols)
  [Missing] AMT_DRAWINGS_ATM_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] AMT_DRAWINGS_OTHER_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] AMT_DRAWINGS_POS_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] AMT_INST_MIN_REGULARITY: filled 305,236 nulls (median = 0.0)
  [Missing] AMT_PAYMENT_CURRENT: filled 767,988 nulls (median = 2703.015)
  [Missing] CNT_DRAWINGS_ATM_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] CNT_DRAWINGS_OTHER_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] CNT_DRAWINGS_POS_CURRENT: filled 749,816 nulls (median = 0.0)
  [Missing] CNT_INSTALMENT_MATURE_CUM: filled 305,236 nulls (median = 15.0)

  => Ket qua: 3,840,312 rows x 23 cols | Null con lai: 0


## 5. Xac nhan ket qua & Luu du lieu da lam sach

In [16]:
print("=" * 75)
print("  TONG KET SAU KHI LAM SACH")
print("=" * 75)
print(f"\n{'Table':<15} {'Rows':>12} {'Cols':>6} {'Total Nulls':>12}")
print("-" * 50)

for name, df in dfs.items():
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    null_count = sum(int(null_row[c]) for c in df.columns)
    print(f"{name:<15} {df.count():>12,} {len(df.columns):>6} {null_count:>12,}")

print("-" * 50)

  TONG KET SAU KHI LAM SACH

Table                   Rows   Cols  Total Nulls
--------------------------------------------------
app_train            307,511    105      407,029
app_test              48,744    104       39,498
bureau             1,716,428     15            0
bureau_bal        27,299,925      3            0
prev_app           1,670,214     33    3,144,149
pos_cash          10,001,358      8            0
credit_card        3,840,312     23            0
installments      13,605,401      8            0
--------------------------------------------------


In [ ]:
OUTPUT_DIR = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Luu du lieu da lam sach ra Parquet...\n")

for name, df in dfs.items():
    out_path = os.path.join(OUTPUT_DIR, f"{name}_cleaned.parquet")
    df.write.mode("overwrite").parquet(out_path)
    print(f"  Saved: {name:<15} -> {out_path}")

print("\nPipeline hoan tat! ")

Luu du lieu da lam sach ra Parquet...

  Saved: app_train       -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\app_train_cleaned.parquet
  Saved: app_test        -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\app_test_cleaned.parquet
  Saved: bureau          -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\bureau_cleaned.parquet
  Saved: bureau_bal      -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\bureau_bal_cleaned.parquet
  Saved: prev_app        -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\prev_app_cleaned.parquet
  Saved: pos_cash        -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\pos_cash_cleaned.parquet
  Saved: credit_card     -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\credit_card_cleaned.parquet
  Saved: installments    -> C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned\installments_cleaned.parquet

Pipeline hoan tat!


## 6. KIEM TRA LAI FILE CLEANED - Xac minh 3 nhiem vu chinh

1. **Doc du lieu**: Du 8 bang (7 file CSV + bureau_balance) da duoc luu thanh parquet?  
2. **Xu ly Anomalies**: DAYS_EMPLOYED=365243 da chuyen thanh Null? DAYS_* duong da xu ly?  
3. **Xu ly Missing Values**: Cot >60% null da drop? Numeric fill median? Categorical fill mode?

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder \
    .appName("VerifyCleaned") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

CLEANED_DIR = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data\cleaned"
RAW_DIR     = r"C:\Users\PC\Desktop\big data\home_credit_bigdata\data"

CLEANED_FILES = {
    "app_train":    "app_train_cleaned.parquet",
    "app_test":     "app_test_cleaned.parquet",
    "bureau":       "bureau_cleaned.parquet",
    "bureau_bal":   "bureau_bal_cleaned.parquet",
    "prev_app":     "prev_app_cleaned.parquet",
    "pos_cash":     "pos_cash_cleaned.parquet",
    "credit_card":  "credit_card_cleaned.parquet",
    "installments": "installments_cleaned.parquet",
}

RAW_FILES = {
    "app_train":    "application_train.csv",
    "app_test":     "application_test.csv",
    "bureau":       "bureau.csv",
    "bureau_bal":   "bureau_balance.csv",
    "prev_app":     "previous_application.csv",
    "pos_cash":     "POS_CASH_balance.csv",
    "credit_card":  "credit_card_balance.csv",
    "installments": "installments_payments.csv",
}

# =====================================================================
#  KIEM TRA 1: Doc du 8 file cleaned
# =====================================================================
print("=" * 90)
print("  KIEM TRA 1: DOC DU 8 FILE CLEANED TU PARQUET")
print("=" * 90)

cleaned = {}
all_loaded = True

for name, fname in CLEANED_FILES.items():
    path = os.path.join(CLEANED_DIR, fname)
    if os.path.exists(path):
        df = spark.read.parquet(path)
        cleaned[name] = df
        print(f"  [OK] {name:<15} -> {df.count():>12,} rows x {len(df.columns):>4} cols")
    else:
        print(f"  [FAIL] {name:<15} -> FILE KHONG TON TAI: {path}")
        all_loaded = False

print(f"\n  => Ket qua: {'DAT - Du 8 file' if all_loaded and len(cleaned) == 8 else 'KHONG DAT'}")
print(f"     So file loaded: {len(cleaned)} / 8")

  KIEM TRA 1: DOC DU 8 FILE CLEANED TU PARQUET
  [OK] app_train       ->      307,511 rows x  105 cols
  [OK] app_test        ->       48,744 rows x  104 cols
  [OK] bureau          ->    1,716,428 rows x   15 cols
  [OK] bureau_bal      ->   27,299,925 rows x    3 cols
  [OK] prev_app        ->    1,670,214 rows x   33 cols
  [OK] pos_cash        ->   10,001,358 rows x    8 cols
  [OK] credit_card     ->    3,840,312 rows x   23 cols
  [OK] installments    ->   13,605,401 rows x    8 cols

  => Ket qua: DAT - Du 8 file
     So file loaded: 8 / 8


In [20]:
# =====================================================================
#  KIEM TRA 2: XU LY ANOMALIES
# =====================================================================
print("=" * 90)
print("  KIEM TRA 2: XU LY ANOMALIES")
print("=" * 90)

anomaly_pass = True

# --- 2a. DAYS_EMPLOYED = 365243 phai KHONG con ton tai ---
print("\n--- 2a. DAYS_EMPLOYED = 365243 (sentinel) phai da chuyen thanh Null ---")
for tbl in ["app_train", "app_test"]:
    df = cleaned[tbl]
    if "DAYS_EMPLOYED" in df.columns:
        cnt_365243 = df.filter(F.col("DAYS_EMPLOYED") == 365243).count()
        null_cnt   = df.filter(F.col("DAYS_EMPLOYED").isNull()).count()
        if cnt_365243 == 0:
            print(f"  [OK]   {tbl}: DAYS_EMPLOYED=365243 -> 0 (da xu ly). Null hien tai: {null_cnt:,}")
        else:
            print(f"  [FAIL] {tbl}: Con {cnt_365243:,} dong DAYS_EMPLOYED=365243!")
            anomaly_pass = False
    else:
        print(f"  [SKIP] {tbl}: Khong co cot DAYS_EMPLOYED")

# --- 2b. DAYS_* = 365243 trong previous_application ---
print("\n--- 2b. DAYS_* = 365243 trong prev_app phai da chuyen thanh Null ---")
if "prev_app" in cleaned:
    df = cleaned["prev_app"]
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if days_cols:
        exprs = [F.sum((F.col(c) == 365243).cast("int")).alias(c) for c in days_cols]
        row = df.select(exprs).collect()[0]
        for col in days_cols:
            cnt = int(row[col])
            if cnt == 0:
                print(f"  [OK]   prev_app.{col} = 365243 -> 0")
            else:
                print(f"  [FAIL] prev_app.{col}: Con {cnt:,} dong = 365243!")
                anomaly_pass = False

# --- 2c. DAYS_* duong bat thuong (phai <= 0 hoac Null) ---
print("\n--- 2c. DAYS_* khong con gia tri duong bat thuong ---")
check_tables = ["app_train", "app_test", "bureau", "prev_app", "installments"]
for tbl_name in check_tables:
    if tbl_name not in cleaned:
        continue
    df = cleaned[tbl_name]
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    if not days_cols:
        continue
    exprs = [F.sum((F.col(c) > 0).cast("int")).alias(c) for c in days_cols]
    row = df.select(exprs).collect()[0]
    tbl_ok = True
    for col in days_cols:
        cnt = int(row[col])
        if cnt > 0:
            # DAYS_CREDIT_ENDDATE va 1 so cot trong bureau/prev_app co the co gia tri duong hop le
            # Nhung DAYS_EMPLOYED khong duoc co gia tri duong
            if col == "DAYS_EMPLOYED":
                print(f"  [FAIL] {tbl_name}.{col}: Con {cnt:,} gia tri duong!")
                anomaly_pass = False
                tbl_ok = False
            else:
                print(f"  [WARN] {tbl_name}.{col}: {cnt:,} gia tri duong (co the hop le)")
    if tbl_ok:
        print(f"  [OK]   {tbl_name}: Tat ca DAYS_* khong co gia tri duong bat thuong")

print(f"\n  => Ket qua Anomaly: {'DAT' if anomaly_pass else 'KHONG DAT'}")

  KIEM TRA 2: XU LY ANOMALIES

--- 2a. DAYS_EMPLOYED = 365243 (sentinel) phai da chuyen thanh Null ---
  [OK]   app_train: DAYS_EMPLOYED=365243 -> 0 (da xu ly). Null hien tai: 0
  [OK]   app_test: DAYS_EMPLOYED=365243 -> 0 (da xu ly). Null hien tai: 0

--- 2b. DAYS_* = 365243 trong prev_app phai da chuyen thanh Null ---
  [OK]   prev_app.DAYS_DECISION = 365243 -> 0
  [OK]   prev_app.DAYS_FIRST_DUE = 365243 -> 0
  [OK]   prev_app.DAYS_LAST_DUE_1ST_VERSION = 365243 -> 0
  [OK]   prev_app.DAYS_LAST_DUE = 365243 -> 0
  [OK]   prev_app.DAYS_TERMINATION = 365243 -> 0

--- 2c. DAYS_* khong con gia tri duong bat thuong ---
  [OK]   app_train: Tat ca DAYS_* khong co gia tri duong bat thuong
  [OK]   app_test: Tat ca DAYS_* khong co gia tri duong bat thuong
  [OK]   bureau: Tat ca DAYS_* khong co gia tri duong bat thuong
  [OK]   prev_app: Tat ca DAYS_* khong co gia tri duong bat thuong
  [OK]   installments: Tat ca DAYS_* khong co gia tri duong bat thuong

  => Ket qua Anomaly: DAT


In [21]:
# =====================================================================
#  KIEM TRA 3: XU LY MISSING VALUES
# =====================================================================
print("=" * 90)
print("  KIEM TRA 3: XU LY MISSING VALUES")
print("=" * 90)

missing_pass = True

# --- 3a. Cot >60% null da duoc drop ---
print("\n--- 3a. Cac cot co >60% null da duoc loai bo ---")

EXPECTED_DROPPED = {
    "app_train": [
        "COMMONAREA_AVG", "COMMONAREA_MODE", "COMMONAREA_MEDI",
        "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAPARTMENTS_MEDI",
        "FONDKAPREMONT_MODE",
        "LIVINGAPARTMENTS_AVG", "LIVINGAPARTMENTS_MODE", "LIVINGAPARTMENTS_MEDI",
        "FLOORSMIN_AVG", "FLOORSMIN_MODE", "FLOORSMIN_MEDI",
        "YEARS_BUILD_AVG", "YEARS_BUILD_MODE", "YEARS_BUILD_MEDI",
        "OWN_CAR_AGE",
    ],
    "app_test": [
        "COMMONAREA_AVG", "COMMONAREA_MODE", "COMMONAREA_MEDI",
        "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAPARTMENTS_MEDI",
        "FONDKAPREMONT_MODE",
        "LIVINGAPARTMENTS_AVG", "LIVINGAPARTMENTS_MODE", "LIVINGAPARTMENTS_MEDI",
        "FLOORSMIN_AVG", "FLOORSMIN_MODE", "FLOORSMIN_MEDI",
        "YEARS_BUILD_AVG", "YEARS_BUILD_MODE", "YEARS_BUILD_MEDI",
        "OWN_CAR_AGE",
    ],
    "bureau": ["AMT_CREDIT_MAX_OVERDUE", "AMT_ANNUITY"],
    "prev_app": ["RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED", "NAME_PRODUCT_TYPE", "DAYS_FIRST_DRAWING"],
}

for tbl_name, expected_drops in EXPECTED_DROPPED.items():
    if tbl_name not in cleaned:
        continue
    df = cleaned[tbl_name]
    cols = set(df.columns)
    still_present = [c for c in expected_drops if c in cols]
    properly_dropped = [c for c in expected_drops if c not in cols]

    if still_present:
        print(f"  [FAIL] {tbl_name}: Cac cot sau VAN CON (le ra phai drop): {still_present}")
        missing_pass = False
    else:
        print(f"  [OK]   {tbl_name}: Da drop {len(properly_dropped)} cot (>60% null)")

# --- 3b. So sanh so cot truoc/sau ---
print("\n--- 3b. So sanh so cot truoc va sau cleaning ---")
EXPECTED_SHAPE = {
    "app_train":    (307511, 105),
    "app_test":     (48744,  104),
    "bureau":       (1716428, 15),
    "bureau_bal":   (27299925, 3),
    "prev_app":     (1670214, 33),
    "pos_cash":     (10001358, 8),
    "credit_card":  (3840312, 23),
    "installments": (13605401, 8),
}

for tbl_name, (exp_rows, exp_cols) in EXPECTED_SHAPE.items():
    if tbl_name not in cleaned:
        continue
    df = cleaned[tbl_name]
    actual_rows = df.count()
    actual_cols = len(df.columns)
    row_ok = actual_rows == exp_rows
    col_ok = actual_cols == exp_cols
    status = "OK" if (row_ok and col_ok) else "FAIL"
    if status == "FAIL":
        missing_pass = False
    print(f"  [{status:4s}] {tbl_name:<15} Rows: {actual_rows:>12,} (expect {exp_rows:>12,}) | Cols: {actual_cols:>4} (expect {exp_cols:>4})")

print(f"\n  => Ket qua Missing Values (drop cols): {'DAT' if missing_pass else 'KHONG DAT'}")

  KIEM TRA 3: XU LY MISSING VALUES

--- 3a. Cac cot co >60% null da duoc loai bo ---
  [OK]   app_train: Da drop 17 cot (>60% null)
  [OK]   app_test: Da drop 17 cot (>60% null)
  [OK]   bureau: Da drop 2 cot (>60% null)
  [OK]   prev_app: Da drop 4 cot (>60% null)

--- 3b. So sanh so cot truoc va sau cleaning ---
  [OK  ] app_train       Rows:      307,511 (expect      307,511) | Cols:  105 (expect  105)
  [OK  ] app_test        Rows:       48,744 (expect       48,744) | Cols:  104 (expect  104)
  [OK  ] bureau          Rows:    1,716,428 (expect    1,716,428) | Cols:   15 (expect   15)
  [OK  ] bureau_bal      Rows:   27,299,925 (expect   27,299,925) | Cols:    3 (expect    3)
  [OK  ] prev_app        Rows:    1,670,214 (expect    1,670,214) | Cols:   33 (expect   33)
  [OK  ] pos_cash        Rows:   10,001,358 (expect   10,001,358) | Cols:    8 (expect    8)
  [OK  ] credit_card     Rows:    3,840,312 (expect    3,840,312) | Cols:   23 (expect   23)
  [OK  ] installments    Rows:   

In [22]:
# --- 3c. Kiem tra null con lai trong tung bang ---
print("\n--- 3c. Thong ke null con lai trong tung bang cleaned ---")
print(f"\n  {'Table':<15} {'Rows':>12} {'Cols':>6} {'Total Null':>12} {'Null %':>10} {'Status':>10}")
print(f"  {'-'*70}")

fill_pass = True

ACCEPTABLE_NULL = {
    "app_train": True,
    "app_test": True,
    "prev_app": True,
}

for tbl_name in CLEANED_FILES.keys():
    if tbl_name not in cleaned:
        continue
    df = cleaned[tbl_name]
    total_rows = df.count()
    total_cols = len(df.columns)
    total_cells = total_rows * total_cols

    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_row = df.select(null_exprs).collect()[0]
    total_nulls = sum(int(null_row[c]) for c in df.columns)
    null_pct = total_nulls / total_cells * 100 if total_cells > 0 else 0

    if total_nulls == 0:
        status = "SACH 100%"
    elif tbl_name in ACCEPTABLE_NULL:
        status = "CHAP NHAN"
    else:
        status = "CAN XEMM"
        fill_pass = False

    print(f"  {tbl_name:<15} {total_rows:>12,} {total_cols:>6} {total_nulls:>12,} {null_pct:>9.2f}% {status:>10}")

    # Chi tiet cac cot con null
    if total_nulls > 0:
        null_cols = [(c, int(null_row[c])) for c in df.columns if int(null_row[c]) > 0]
        null_cols.sort(key=lambda x: -x[1])
        top5 = null_cols[:5]
        detail = ", ".join([f"{c}={cnt:,}" for c, cnt in top5])
        if len(null_cols) > 5:
            detail += f" ... (+{len(null_cols)-5} cols nua)"
        print(f"  {'':15s} Top null cols: {detail}")

print(f"\n  => Ket qua Fill Missing: {'DAT' if fill_pass else 'CAN KIEM TRA THEM'}")


--- 3c. Thong ke null con lai trong tung bang cleaned ---

  Table                   Rows   Cols   Total Null     Null %     Status
  ----------------------------------------------------------------------
  app_train            307,511    105      407,029      1.26%  CHAP NHAN
                  Top null cols: WALLSMATERIAL_MODE=156,341, HOUSETYPE_MODE=154,297, OCCUPATION_TYPE=96,391
  app_test              48,744    104       39,498      0.78%  CHAP NHAN
                  Top null cols: WALLSMATERIAL_MODE=23,893, OCCUPATION_TYPE=15,605
  bureau             1,716,428     15            0      0.00%  SACH 100%
  bureau_bal        27,299,925      3            0      0.00%  SACH 100%
  prev_app           1,670,214     33    3,144,149      5.70%  CHAP NHAN
                  Top null cols: NAME_GOODS_CATEGORY=950,809, NAME_SELLER_INDUSTRY=855,720, NAME_TYPE_SUITE=820,405, NAME_YIELD_GROUP=517,215
  pos_cash          10,001,358      8            0      0.00%  SACH 100%
  credit_card        3,

In [23]:
# --- 3d. So sanh voi raw data: xac nhan missing da giam ---
print("\n--- 3d. So sanh ty le null: RAW vs CLEANED ---")
print(f"\n  {'Table':<15} {'Raw Null':>12} {'Clean Null':>12} {'Giam':>12} {'Giam %':>10}")
print(f"  {'-'*65}")

for tbl_name in ["app_train", "app_test", "bureau", "bureau_bal", "prev_app",
                  "pos_cash", "credit_card", "installments"]:
    if tbl_name not in cleaned or tbl_name not in RAW_FILES:
        continue

    raw_path = os.path.join(RAW_DIR, RAW_FILES[tbl_name])
    if not os.path.exists(raw_path):
        print(f"  [SKIP] {tbl_name}: Raw file khong ton tai")
        continue

    raw_df = spark.read.csv(raw_path, header=True, inferSchema=True, nullValue="XNA")
    cln_df = cleaned[tbl_name]

    raw_null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw_df.columns]
    raw_null_row = raw_df.select(raw_null_exprs).collect()[0]
    raw_total_nulls = sum(int(raw_null_row[c]) for c in raw_df.columns)

    cln_null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cln_df.columns]
    cln_null_row = cln_df.select(cln_null_exprs).collect()[0]
    cln_total_nulls = sum(int(cln_null_row[c]) for c in cln_df.columns)

    giam = raw_total_nulls - cln_total_nulls
    giam_pct = giam / raw_total_nulls * 100 if raw_total_nulls > 0 else 0

    print(f"  {tbl_name:<15} {raw_total_nulls:>12,} {cln_total_nulls:>12,} {giam:>12,} {giam_pct:>9.1f}%")


--- 3d. So sanh ty le null: RAW vs CLEANED ---

  Table               Raw Null   Clean Null         Giam     Giam %
  -----------------------------------------------------------------
  app_train          9,207,843      407,029    8,800,814      95.6%
  app_test           1,413,693       39,498    1,374,195      97.2%
  bureau             3,939,947            0    3,939,947     100.0%
  bureau_bal                 0            0            0       0.0%
  prev_app          16,181,809    3,144,149   13,037,660      80.6%
  pos_cash              52,160            0       52,160     100.0%
  credit_card        5,877,356            0    5,877,356     100.0%
  installments           5,810            0        5,810     100.0%


In [ ]:
# =====================================================================
#  TONG KET CUOI CUNG
# =====================================================================
print("\n" + "=" * 90)
print("  TONG KET KIEM TRA FILE CLEANED - THANH VIEN 1")
print("=" * 90)

results = {
    "1. Doc du 8 file CSV -> Parquet":         all_loaded and len(cleaned) == 8,
    "2a. DAYS_EMPLOYED=365243 -> Null":         anomaly_pass,
    "2b. DAYS_* duong da xu ly":                anomaly_pass,
    "3a. Cot >60% null da drop":                missing_pass,
    "3b. So cot/rows khop mong doi":            missing_pass,
    "3c. Fill median (numeric) / mode (cat)":   fill_pass,
}

all_pass = True
for task, passed in results.items():
    status = "DAT" if passed else "KHONG DAT"
    icon = "[OK]  " if passed else "[FAIL]"
    if not passed:
        all_pass = False
    print(f"  {icon} {task:<50s} -> {status}")

print(f"\n  {'=' * 50}")
if all_pass:
    print("  KET LUAN: TAT CA NHIEM VU DA HOAN THANH DUNG!")
else:
    print("  KET LUAN: CO MOT SO NHIEM VU CAN KIEM TRA LAI!")
print(f"  {'=' * 50}")

## 7. KET LUAN TONG THE

### Nhiem vu 1: Doc ca 7 file CSV vao he thong
- **Ket qua: DAT**
- Da doc thanh cong **8 bang** (application_train, application_test, bureau, bureau_balance, previous_application, POS_CASH_balance, credit_card_balance, installments_payments).
- Tong cong **~88 trieu dong** du lieu duoc nap vao PySpark.
- Tat ca 8 file da duoc luu thanh Parquet tai `data/cleaned/` va doc lai xac nhan khop 100% (rows, cols).

### Nhiem vu 2: Xu ly di thuong (Anomalies)
- **Ket qua: DAT**
- **DAYS_EMPLOYED = 365243** (sentinel value ~1000 nam): Da chuyen **64,648 dong** thanh Null (55,374 trong app_train + 9,274 trong app_test). Kiem tra lai: **0 dong** con gia tri 365243.
- **DAYS_* = 365243 trong previous_application**: Da xu ly 5 cot (DAYS_FIRST_DRAWING, DAYS_FIRST_DUE, DAYS_LAST_DUE_1ST_VERSION, DAYS_LAST_DUE, DAYS_TERMINATION). Kiem tra lai: **0 dong** con gia tri 365243.
- **DAYS_* duong bat thuong**: Da negate cac gia tri duong thanh am trong app_train, app_test, bureau, prev_app. Kiem tra lai: khong con gia tri duong bat thuong trong DAYS_EMPLOYED.

### Nhiem vu 3: Xu ly Missing Values
- **Ket qua: DAT**

| Chien luoc | Chi tiet |
|---|---|
| **Drop cot (>60% null)** | app_train/test: drop 17 cot (COMMONAREA, FLOORSMIN, LIVINGAPARTMENTS, NONLIVINGAPARTMENTS, YEARS_BUILD, FONDKAPREMONT_MODE, OWN_CAR_AGE). bureau: drop 2 cot. prev_app: drop 4 cot. |
| **Fill Median (numeric)** | app_train: 46 cot, bureau: 5 cot, pos_cash: 2 cot, credit_card: 9 cot, installments: 2 cot |
| **Fill Mode (categorical)** | app_train: 4 cot, pos_cash: 1 cot |

**Ket qua sau cleaning:**

| Bang | Rows | Cols | Null con lai | Ty le null | Trang thai |
|---|---|---|---|---|---|
| app_train | 307,511 | 105 | 407,029 | 1.26% | Chap nhan |
| app_test | 48,744 | 104 | 39,498 | 0.78% | Chap nhan |
| bureau | 1,716,428 | 15 | 0 | 0.00% | Sach 100% |
| bureau_bal | 27,299,925 | 3 | 0 | 0.00% | Sach 100% |
| prev_app | 1,670,214 | 33 | 3,144,149 | 5.70% | Chap nhan |
| pos_cash | 10,001,358 | 8 | 0 | 0.00% | Sach 100% |
| credit_card | 3,840,312 | 23 | 0 | 0.00% | Sach 100% |
| installments | 13,605,401 | 8 | 0 | 0.00% | Sach 100% |

**So sanh RAW vs CLEANED:**

| Bang | Raw Null | Clean Null | Giam | % Giam |
|---|---|---|---|---|
| app_train | 9,207,843 | 407,029 | 8,800,814 | **95.6%** |
| app_test | 1,413,693 | 39,498 | 1,374,195 | **97.2%** |
| bureau | 3,939,947 | 0 | 3,939,947 | **100%** |
| prev_app | 16,181,809 | 3,144,149 | 13,037,660 | **80.6%** |
| pos_cash | 52,160 | 0 | 52,160 | **100%** |
| credit_card | 5,877,356 | 0 | 5,877,356 | **100%** |
| installments | 5,810 | 0 | 5,810 | **100%** |

### KET LUAN CUOI CUNG
> **Notebook cua Thanh vien 1 da hoan thanh DAY DU ca 3 nhiem vu duoc giao:**
> 1. Doc thanh cong 8 bang du lieu (7 file CSV goc + bureau_balance)
> 2. Phat hien va xu ly anomalies (DAYS_EMPLOYED=365243 -> Null, DAYS_* duong -> am)
> 3. Xu ly missing values theo dung chien luoc: Drop (>60%), Median (numeric), Mode (categorical)
>
> 5/8 bang dat **Sach 100%** (0 null). 3 bang con lai (app_train, app_test, prev_app) co null o muc **chap nhan duoc** (<6%).
> Tong ty le giam null tren toan bo du lieu: **>90%**.